# Clean-hard targeted training vs random control

RQ1: does up-weighting hard rows with a clear label beat up-weighting the same number of random rows? "Hard" is tested two ways: rows the baseline is **unsure** about (the 0.4-0.6 band) and rows it got **wrong** out-of-fold (Just Train Twice, Liu et al. 2021).
RQ2: does the answer change when the FinBERT embeddings are fine-tuned instead of frozen?

Every run retrains the FinBERT CNN from scratch with the baseline recipe; only the loss weights change.

| arm | rows weighted K (all others 1) |
|---|---|
| `no_weight` | none, the reference |
| `clean_hard` | 103 hard + all-agree rows |
| `random` | 103 random train rows (new draw per seed) |
| `clean_random` | 103 all-agree, non-hard rows (new draw per seed) |
| `variant` | clean-hard rows get K, the 261 hard rows without agreement get 0.5 |
| `clean_error` | 114 out-of-fold errors that are all-agree (Just Train Twice) |
| `error_random` | 114 random train rows (new draw per seed) |
| `error_clean_random` | 114 all-agree rows the baseline got right (new draw per seed) |

Row sets come from `results/rq1_rq2/row_sets.csv`, scores are on **val** only. K is chosen here; test is run once at the end.
Every run goes to `results/rq1_rq2/runs.csv` (re-running a run replaces it) and its val probabilities to `results/rq1_rq2/probs/` for the taxonomy.

## Imports

In [1]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
import sys

import numpy as np
import pandas as pd

sys.path.append("..")
from utils.clean_hard import run_arm, save_runs
from utils.finbert_cnn import SEEDS, load_finbert_inputs
from utils.preprocessing import encode_labels

## Data

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
X_train, X_val, embedding_matrix = load_finbert_inputs(train_df, val_df)

row_sets = pd.read_csv("../results/rq1_rq2/row_sets.csv")
data = dict(embedding_matrix=embedding_matrix, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
            band_val=pd.read_csv("../results/hard_samples/band_val.csv"))

RUNS_PATH = "../results/rq1_rq2/runs.csv"
PROBS_DIR = "../results/rq1_rq2/probs"
os.makedirs(PROBS_DIR, exist_ok=True)
KS = [3, 5, 10]
KS_ERROR = [5, 10, 20]   # Just Train Twice uses larger weights
print(row_sets.groupby("row_set").train_idx.count() // len(SEEDS))

/Users/cleojj002/Documents/GitHub/nlp-financial-phrasebank/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 16:53:05.160000 52018 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36959.95it/s]
[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | U

row_set
ambiguous_hard        261
clean_error           114
clean_hard            103
clean_random          103
error_clean_random    114
error_random          114
random                103
Name: train_idx, dtype: int64


In [3]:
def run_seeds(arm, k, trainable=False, epochs=20):
    rows = []
    for seed in SEEDS:
        row, probs = run_arm(arm, k, seed, data, row_sets, trainable=trainable, epochs=epochs)
        rows.append(row)
        name = f"{arm}_k{row['k']}_{row['embedding_config']}_s{seed}"
        np.save(f"{PROBS_DIR}/{name}.npy", probs)
        print(f"{arm} k={row['k']} seed={seed}: f1={row['f1_overall']:.3f}  hard={row['f1_hard']:.3f}")
    return save_runs(RUNS_PATH, pd.DataFrame(rows))


## 1. No-weight reference (run this first)

Check: every seed should land around 0.74-0.77. A seed at about 0.248 means it got stuck predicting neutral; stop and fix that before running the other arms.

In [4]:
runs = run_seeds("no_weight", 1)

 - val_macro_f1: 0.3184
 - val_macro_f1: 0.4151
 - val_macro_f1: 0.4368
 - val_macro_f1: 0.5897
 - val_macro_f1: 0.6485
 - val_macro_f1: 0.6796
 - val_macro_f1: 0.7090
 - val_macro_f1: 0.7188
 - val_macro_f1: 0.7110
 - val_macro_f1: 0.7303
 - val_macro_f1: 0.7102
 - val_macro_f1: 0.7318
 - val_macro_f1: 0.7275
 - val_macro_f1: 0.7436
 - val_macro_f1: 0.7451
 - val_macro_f1: 0.7605
 - val_macro_f1: 0.7653
 - val_macro_f1: 0.7455
 - val_macro_f1: 0.7468
 - val_macro_f1: 0.7354
no_weight k=1 seed=42: f1=0.765  hard=0.606
 - val_macro_f1: 0.3505
 - val_macro_f1: 0.4550
 - val_macro_f1: 0.6215
 - val_macro_f1: 0.6913
 - val_macro_f1: 0.7415
 - val_macro_f1: 0.7549
 - val_macro_f1: 0.7519
 - val_macro_f1: 0.7570
 - val_macro_f1: 0.7603
 - val_macro_f1: 0.7525
 - val_macro_f1: 0.7608
 - val_macro_f1: 0.7498
 - val_macro_f1: 0.7576
 - val_macro_f1: 0.7508
 - val_macro_f1: 0.7296
 - val_macro_f1: 0.7423
no_weight k=1 seed=43: f1=0.761  hard=0.617
 - val_macro_f1: 0.3246
 - val_macro_f1: 0.4144


## 2. Targeted arms (frozen embeddings)

In [5]:
for k in KS:
    runs = run_seeds("clean_hard", k)

 - val_macro_f1: 0.3875
 - val_macro_f1: 0.4194
 - val_macro_f1: 0.4652
 - val_macro_f1: 0.6428
 - val_macro_f1: 0.6661
 - val_macro_f1: 0.6761
 - val_macro_f1: 0.7226
 - val_macro_f1: 0.7182
 - val_macro_f1: 0.7428
 - val_macro_f1: 0.7184
 - val_macro_f1: 0.7336
 - val_macro_f1: 0.7353
 - val_macro_f1: 0.7295
 - val_macro_f1: 0.7653
 - val_macro_f1: 0.7545
 - val_macro_f1: 0.7476
 - val_macro_f1: 0.7524
 - val_macro_f1: 0.7522
 - val_macro_f1: 0.7329
clean_hard k=3 seed=42: f1=0.765  hard=0.571
 - val_macro_f1: 0.2950
 - val_macro_f1: 0.4653
 - val_macro_f1: 0.6276
 - val_macro_f1: 0.7039
 - val_macro_f1: 0.7368
 - val_macro_f1: 0.7485
 - val_macro_f1: 0.7626
 - val_macro_f1: 0.7634
 - val_macro_f1: 0.7659
 - val_macro_f1: 0.7573
 - val_macro_f1: 0.7686
 - val_macro_f1: 0.7375
 - val_macro_f1: 0.7386
 - val_macro_f1: 0.7409
 - val_macro_f1: 0.7217
 - val_macro_f1: 0.7010
clean_hard k=3 seed=43: f1=0.769  hard=0.587
 - val_macro_f1: 0.3154
 - val_macro_f1: 0.4474
 - val_macro_f1: 0.488

In [6]:
for k in KS:
    runs = run_seeds("variant", k)

 - val_macro_f1: 0.3859
 - val_macro_f1: 0.4130
 - val_macro_f1: 0.4539
 - val_macro_f1: 0.5859
 - val_macro_f1: 0.6447
 - val_macro_f1: 0.6678
 - val_macro_f1: 0.7091
 - val_macro_f1: 0.7078
 - val_macro_f1: 0.7119
 - val_macro_f1: 0.7224
 - val_macro_f1: 0.7249
 - val_macro_f1: 0.7248
 - val_macro_f1: 0.7318
 - val_macro_f1: 0.7240
 - val_macro_f1: 0.7293
 - val_macro_f1: 0.7424
 - val_macro_f1: 0.7330
 - val_macro_f1: 0.7374
 - val_macro_f1: 0.7410
 - val_macro_f1: 0.7345
variant k=3 seed=42: f1=0.742  hard=0.482
 - val_macro_f1: 0.2973
 - val_macro_f1: 0.4561
 - val_macro_f1: 0.5698
 - val_macro_f1: 0.6557
 - val_macro_f1: 0.7413
 - val_macro_f1: 0.7406
 - val_macro_f1: 0.7577
 - val_macro_f1: 0.7687
 - val_macro_f1: 0.7599
 - val_macro_f1: 0.7667
 - val_macro_f1: 0.7540
 - val_macro_f1: 0.7297
 - val_macro_f1: 0.7408
variant k=3 seed=43: f1=0.769  hard=0.569
 - val_macro_f1: 0.3603
 - val_macro_f1: 0.4439
 - val_macro_f1: 0.5008
 - val_macro_f1: 0.5793
 - val_macro_f1: 0.6397
 - v

## 3. Controls (frozen embeddings)

In [7]:
for k in KS:
    runs = run_seeds("random", k)

 - val_macro_f1: 0.3813
 - val_macro_f1: 0.4151
 - val_macro_f1: 0.4477
 - val_macro_f1: 0.6264
 - val_macro_f1: 0.6616
 - val_macro_f1: 0.6846
 - val_macro_f1: 0.6881
 - val_macro_f1: 0.7232
 - val_macro_f1: 0.7217
 - val_macro_f1: 0.7157
 - val_macro_f1: 0.7315
 - val_macro_f1: 0.7434
 - val_macro_f1: 0.7243
 - val_macro_f1: 0.7351
 - val_macro_f1: 0.7383
 - val_macro_f1: 0.7466
 - val_macro_f1: 0.7364
 - val_macro_f1: 0.7347
 - val_macro_f1: 0.7462
 - val_macro_f1: 0.7198
random k=3 seed=42: f1=0.747  hard=0.485
 - val_macro_f1: 0.3350
 - val_macro_f1: 0.4917
 - val_macro_f1: 0.6499
 - val_macro_f1: 0.6803
 - val_macro_f1: 0.7198
 - val_macro_f1: 0.7413
 - val_macro_f1: 0.7388
 - val_macro_f1: 0.7471
 - val_macro_f1: 0.7582
 - val_macro_f1: 0.7431
 - val_macro_f1: 0.7486
 - val_macro_f1: 0.7195
 - val_macro_f1: 0.7239
 - val_macro_f1: 0.7367
random k=3 seed=43: f1=0.758  hard=0.622
 - val_macro_f1: 0.3306
 - val_macro_f1: 0.4159
 - val_macro_f1: 0.4419
 - val_macro_f1: 0.4974
 - val

In [8]:
for k in KS:
    runs = run_seeds("clean_random", k)

 - val_macro_f1: 0.3789
 - val_macro_f1: 0.4167
 - val_macro_f1: 0.4383
 - val_macro_f1: 0.5355
 - val_macro_f1: 0.6523
 - val_macro_f1: 0.6658
 - val_macro_f1: 0.7104
 - val_macro_f1: 0.7206
 - val_macro_f1: 0.7105
 - val_macro_f1: 0.7178
 - val_macro_f1: 0.7286
 - val_macro_f1: 0.7260
 - val_macro_f1: 0.7107
 - val_macro_f1: 0.7264
 - val_macro_f1: 0.7353
 - val_macro_f1: 0.7522
 - val_macro_f1: 0.7475
 - val_macro_f1: 0.7393
 - val_macro_f1: 0.7366
 - val_macro_f1: 0.7351
clean_random k=3 seed=42: f1=0.752  hard=0.566
 - val_macro_f1: 0.3362
 - val_macro_f1: 0.4335
 - val_macro_f1: 0.6169
 - val_macro_f1: 0.6845
 - val_macro_f1: 0.7218
 - val_macro_f1: 0.7475
 - val_macro_f1: 0.7510
 - val_macro_f1: 0.7534
 - val_macro_f1: 0.7607
 - val_macro_f1: 0.7514
 - val_macro_f1: 0.7452
 - val_macro_f1: 0.7338
 - val_macro_f1: 0.7329
 - val_macro_f1: 0.7235
clean_random k=3 seed=43: f1=0.761  hard=0.530
 - val_macro_f1: 0.2754
 - val_macro_f1: 0.4020
 - val_macro_f1: 0.4873
 - val_macro_f1: 0

## 3b. Just Train Twice: up-weight the baseline's errors (frozen embeddings)

The targeted rows are the 114 train sentences the out-of-fold CNN got wrong and all annotators agreed on, so label noise is filtered out. Unlike the band, this includes confident errors (72 of the 114).

In [9]:
for k in KS_ERROR:
    runs = run_seeds("clean_error", k)

 - val_macro_f1: 0.3857
 - val_macro_f1: 0.4305
 - val_macro_f1: 0.4448
 - val_macro_f1: 0.4635
 - val_macro_f1: 0.4881
 - val_macro_f1: 0.5779
 - val_macro_f1: 0.6581
 - val_macro_f1: 0.6785
 - val_macro_f1: 0.6895
 - val_macro_f1: 0.6977
 - val_macro_f1: 0.7028
 - val_macro_f1: 0.7216
 - val_macro_f1: 0.7261
 - val_macro_f1: 0.7282
 - val_macro_f1: 0.7283
 - val_macro_f1: 0.7329
 - val_macro_f1: 0.7334
 - val_macro_f1: 0.7445
 - val_macro_f1: 0.7468
 - val_macro_f1: 0.7373
clean_error k=5 seed=42: f1=0.747  hard=0.412
 - val_macro_f1: 0.3314
 - val_macro_f1: 0.3908
 - val_macro_f1: 0.4790
 - val_macro_f1: 0.5982
 - val_macro_f1: 0.6987
 - val_macro_f1: 0.7309
 - val_macro_f1: 0.7318
 - val_macro_f1: 0.7472
 - val_macro_f1: 0.7423
 - val_macro_f1: 0.7442
 - val_macro_f1: 0.7391
 - val_macro_f1: 0.7240
 - val_macro_f1: 0.7222
clean_error k=5 seed=43: f1=0.747  hard=0.603
 - val_macro_f1: 0.3471
 - val_macro_f1: 0.3999
 - val_macro_f1: 0.4840
 - val_macro_f1: 0.5560
 - val_macro_f1: 0.6

In [10]:
for k in KS_ERROR:
    runs = run_seeds("error_random", k)

 - val_macro_f1: 0.3364
 - val_macro_f1: 0.4046
 - val_macro_f1: 0.4273
 - val_macro_f1: 0.5186
 - val_macro_f1: 0.6315
 - val_macro_f1: 0.6866
 - val_macro_f1: 0.7209
 - val_macro_f1: 0.7337
 - val_macro_f1: 0.7383
 - val_macro_f1: 0.7277
 - val_macro_f1: 0.7427
 - val_macro_f1: 0.7404
 - val_macro_f1: 0.7255
 - val_macro_f1: 0.7413
 - val_macro_f1: 0.7598
 - val_macro_f1: 0.7570
 - val_macro_f1: 0.7531
 - val_macro_f1: 0.7580
 - val_macro_f1: 0.7552
 - val_macro_f1: 0.7250
error_random k=5 seed=42: f1=0.760  hard=0.590
 - val_macro_f1: 0.3101
 - val_macro_f1: 0.3815
 - val_macro_f1: 0.4767
 - val_macro_f1: 0.6453
 - val_macro_f1: 0.7060
 - val_macro_f1: 0.7392
 - val_macro_f1: 0.7537
 - val_macro_f1: 0.7607
 - val_macro_f1: 0.7531
 - val_macro_f1: 0.7531
 - val_macro_f1: 0.7676
 - val_macro_f1: 0.7176
 - val_macro_f1: 0.7397
 - val_macro_f1: 0.7302
 - val_macro_f1: 0.7077
 - val_macro_f1: 0.7328
error_random k=5 seed=43: f1=0.768  hard=0.528
 - val_macro_f1: 0.2483
 - val_macro_f1: 0

In [11]:
for k in KS_ERROR:
    runs = run_seeds("error_clean_random", k)

 - val_macro_f1: 0.2637
 - val_macro_f1: 0.4078
 - val_macro_f1: 0.4570
 - val_macro_f1: 0.6174
 - val_macro_f1: 0.6732
 - val_macro_f1: 0.6893
 - val_macro_f1: 0.7124
 - val_macro_f1: 0.7239
 - val_macro_f1: 0.7309
 - val_macro_f1: 0.7266
 - val_macro_f1: 0.7279
 - val_macro_f1: 0.7477
 - val_macro_f1: 0.7241
 - val_macro_f1: 0.7537
 - val_macro_f1: 0.7640
 - val_macro_f1: 0.7634
 - val_macro_f1: 0.7465
 - val_macro_f1: 0.7341
 - val_macro_f1: 0.7234
 - val_macro_f1: 0.7410
error_clean_random k=5 seed=42: f1=0.764  hard=0.475
 - val_macro_f1: 0.3557
 - val_macro_f1: 0.4760
 - val_macro_f1: 0.5754
 - val_macro_f1: 0.6768
 - val_macro_f1: 0.7188
 - val_macro_f1: 0.7504
 - val_macro_f1: 0.7395
 - val_macro_f1: 0.7501
 - val_macro_f1: 0.7600
 - val_macro_f1: 0.7478
 - val_macro_f1: 0.7405
 - val_macro_f1: 0.7325
 - val_macro_f1: 0.7501
 - val_macro_f1: 0.7517
error_clean_random k=5 seed=43: f1=0.760  hard=0.548
 - val_macro_f1: 0.2483
 - val_macro_f1: 0.3871
 - val_macro_f1: 0.4251
 - val

## 4. Compare on val and choose K

Mean and spread over the seeds, and the difference to the no-weight reference. Pick K (one for each definition of hard) where targeted minus its controls is largest **compared with the spread**; a difference smaller than the std is not an effect.

In [12]:
runs = pd.read_csv(RUNS_PATH)
cols = ["f1_overall", "f1_hard", "f1_nonhard", "f1_allagree", "f1_not_allagree"]

summary = runs.groupby(["embedding_config", "arm", "k"])[cols].agg(["mean", "std"]).round(3)
summary

f1_overall        f1_hard         \
                                             mean    std    mean    std   
embedding_config arm                k                                     
finbert_frozen   clean_error        5       0.753  0.008   0.521  0.073   
                                    10      0.740  0.007   0.531  0.052   
                                    20      0.725  0.017   0.503  0.067   
                 clean_hard         3       0.764  0.006   0.552  0.042   
                                    5       0.761  0.010   0.518  0.039   
                                    10      0.754  0.008   0.529  0.049   
                 clean_random       3       0.762  0.008   0.515  0.077   
                                    5       0.759  0.011   0.533  0.032   
                                    10      0.757  0.010   0.530  0.054   
                 error_clean_random 5       0.759  0.016   0.510  0.067   
                                    10      0.760  0.007   0.536  0.032   
                                    20      0.748  0.007   0.512  0.020   
                 error_random       5       0.759  0.006   0.492  0.073   
                                    10      0.744  0.005   0.516  0.054   
                                    20      0.737  0.009   0.498  0.044   
                 no_weight          1       0.758  0.005   0.548  0.063   
                 random             3       0.759  0.008   0.516  0.065   
                                    5       0.756  0.006   0.506  0.069   
                                    10      0.748  0.016   0.507  0.079   
                 variant            3       0.756  0.010   0.508  0.037   
                                    5       0.755  0.010   0.533  0.074   
                                    10      0.749  0.006   0.516  0.079   

                                       f1_nonhard        f1_allagree         \
                                             mean    std        mean    std   
embedding_config arm                k                                         
finbert_frozen   clean_error        5       0.766  0.011       0.904  0.015   
                                    10      0.751  0.005       0.900  0.015   
                                    20      0.737  0.014       0.885  0.021   
                 clean_hard         3       0.774  0.004       0.912  0.007   
                                    5       0.772  0.009       0.912  0.008   
                                    10      0.767  0.007       0.910  0.012   
                 clean_random       3       0.774  0.009       0.911  0.008   
                                    5       0.772  0.009       0.915  0.008   
                                    10      0.769  0.011       0.911  0.013   
                 error_clean_random 5       0.770  0.013       0.911  0.010   
                                    10      0.770  0.009       0.919  0.008   
                                    20      0.760  0.008       0.903  0.003   
                 error_random       5       0.772  0.008       0.905  0.011   
                                    10      0.757  0.004       0.899  0.013   
                                    20      0.750  0.009       0.896  0.007   
                 no_weight          1       0.769  0.004       0.909  0.005   
                 random             3       0.771  0.011       0.909  0.009   
                                    5       0.768  0.003       0.906  0.006   
                                    10      0.762  0.014       0.904  0.017   
                 variant            3       0.771  0.009       0.914  0.009   
                                    5       0.765  0.009       0.909  0.011   
                                    10      0.762  0.008       0.908  0.008   

                                       f1_not_allagree         
                                                  mean    std  
embedding_config arm                k                      

In [13]:
frozen = runs[runs.embedding_config == "finbert_frozen"]
reference = frozen[frozen.arm == "no_weight"][cols].mean()

# how much each arm gains over no_weight (mean over seeds)
gain = frozen[frozen.arm != "no_weight"].groupby(["arm", "k"])[cols].mean() - reference
gain.round(3)


f1_overall  f1_hard  f1_nonhard  f1_allagree  \
arm                k                                                  
clean_error        5       -0.005   -0.027      -0.004       -0.005   
                   10      -0.018   -0.017      -0.018       -0.010   
                   20      -0.034   -0.045      -0.033       -0.024   
clean_hard         3        0.006    0.004       0.005        0.003   
                   5        0.002   -0.030       0.003        0.003   
                   10      -0.005   -0.019      -0.003        0.001   
clean_random       3        0.003   -0.033       0.004        0.002   
                   5        0.001   -0.015       0.003        0.006   
                   10      -0.001   -0.018       0.000        0.002   
error_clean_random 5        0.000   -0.037       0.001        0.002   
                   10       0.001   -0.012       0.001        0.009   
                   20      -0.011   -0.036      -0.009       -0.007   
error_random       5        0.001   -0.056       0.003       -0.004   
                   10      -0.014   -0.032      -0.012       -0.010   
                   20      -0.021   -0.049      -0.019       -0.013   
random             3        0.000   -0.032       0.001        0.000   
                   5       -0.002   -0.042      -0.001       -0.003   
                   10      -0.010   -0.041      -0.008       -0.005   
variant            3       -0.003   -0.040       0.002        0.005   
                   5       -0.003   -0.015      -0.004        0.000   
                   10      -0.010   -0.032      -0.007       -0.001   

                       f1_not_allagree  
arm                k                    
clean_error        5            -0.005  
                   10           -0.028  
                   20           -0.045  
clean_hard         3             0.011  
                   5             0.004  
                   10           -0.010  
clean_random       3             0.009  
                   5            -0.006  
                   10           -0.004  
error_clean_random 5             0.000  
                   10           -0.011  
                   20           -0.016  
error_random       5             0.007  
                   10           -0.019  
                   20           -0.032  
random             3             0.001  
                   5             0.002  
                   10           -0.015  
variant            3            -0.012  
                   5            -0.008  
                   10           -0.021

In [14]:
# targeted minus its control, seed by seed (both arms use the same seeds)
pairs = [("clean_hard", "random"), ("clean_hard", "clean_random"),
         ("clean_error", "error_random"), ("clean_error", "error_clean_random")]

for target, control in pairs:
    for k in sorted(frozen[frozen.arm == target].k.unique()):
        a = frozen[(frozen.arm == target) & (frozen.k == k)].sort_values("seed").f1_overall.values
        b = frozen[(frozen.arm == control) & (frozen.k == k)].sort_values("seed").f1_overall.values
        if len(a) > 0 and len(a) == len(b):
            diff = a - b
            print(f"{target} - {control}, k={k}: mean {diff.mean():.3f}, std {diff.std(ddof=1):.3f}, "
                  f"min {diff.min():.3f}, max {diff.max():.3f}")


clean_hard - random
     mean    std    min    max
k                             
3   0.005  0.011 -0.007  0.019
5   0.005  0.011 -0.009  0.016
10  0.005  0.016 -0.017  0.028

clean_hard - clean_random
     mean    std    min    max
k                             
3   0.002  0.010 -0.009  0.013
5   0.002  0.011 -0.017  0.011
10 -0.004  0.012 -0.023  0.010

clean_error - error_random
     mean    std    min    max
k                             
5  -0.006  0.013 -0.020  0.011
10 -0.004  0.005 -0.011  0.001
20 -0.012  0.020 -0.038  0.009

clean_error - error_clean_random
     mean    std    min    max
k                             
5  -0.006  0.018 -0.023  0.020
10 -0.019  0.010 -0.035 -0.009
20 -0.023  0.020 -0.049 -0.004



## 5. RQ2: fine-tuned embeddings at the chosen K

Run the targeted arm that did best in section 4 (clean-hard or clean-error) with its random control. Fine-tuned embeddings learn slowly at lr 5e-5, so they get 80 epochs.

In [ ]:
TARGET, CONTROL = "clean_hard", "random"   # or "clean_error", "error_random"
CHOSEN_K = 5    # fill in after section 4
assert CHOSEN_K in (KS if TARGET == "clean_hard" else KS_ERROR), "choose K in section 4 first"

for arm, k in [("no_weight", 1), (TARGET, CHOSEN_K), (CONTROL, CHOSEN_K)]:
    runs = run_seeds(arm, k, trainable=True, epochs=80)


 - val_macro_f1: 0.3910
 - val_macro_f1: 0.5998
 - val_macro_f1: 0.7122
 - val_macro_f1: 0.7239
 - val_macro_f1: 0.6888
 - val_macro_f1: 0.6946
 - val_macro_f1: 0.6908
 - val_macro_f1: 0.6889
 - val_macro_f1: 0.6958
no_weight k=1 seed=42: f1=0.724  hard=0.426
 - val_macro_f1: 0.4962
 - val_macro_f1: 0.6575
 - val_macro_f1: 0.6949
 - val_macro_f1: 0.7094
 - val_macro_f1: 0.7057
 - val_macro_f1: 0.7035
 - val_macro_f1: 0.7204
 - val_macro_f1: 0.7085
 - val_macro_f1: 0.7131
 - val_macro_f1: 0.7132
 - val_macro_f1: 0.7117
